# Mutual Fund Portfolio Planning using Python
**Project Objective:** Build a mutual-fund style portfolio using historical stock prices, analyze risk and return, and simulate SIP investment growth.

**Libraries Used:** pandas, numpy, plotly


## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

## 2. Load Dataset

In [ ]:
data = pd.read_csv('nifty50_closing_prices.csv')
data['Date'] = pd.to_datetime(data['Date'])
data = data.sort_values('Date')
data.head()

## 3. Data Cleaning

In [ ]:
data = data.fillna(method='ffill')
data.isnull().sum()

## 4. Calculate Daily Returns

In [ ]:
returns = data.set_index('Date').pct_change()
returns = returns.dropna()
returns.head()

## 5. Calculate Annual ROI

In [ ]:
annual_return = returns.mean() * 252 * 100
annual_return.sort_values(ascending=False)

## 6. Calculate Volatility (Risk)

In [ ]:
volatility = returns.std() * np.sqrt(252) * 100
volatility.sort_values()

## 7. Risk vs Return Data

In [ ]:
risk_return_df = pd.DataFrame({
    'Company': annual_return.index,
    'Return (%)': annual_return.values,
    'Volatility (%)': volatility.values
})
risk_return_df.head()

## 8. Apply Threshold Filtering

In [ ]:
roi_threshold = np.percentile(annual_return,60)
volatility_threshold = np.percentile(volatility,40)

selected_companies = annual_return[
    (annual_return > roi_threshold) &
    (volatility < volatility_threshold)
]

selected_companies = selected_companies.sort_values(ascending=False)
selected_companies

## 9. Portfolio Bar Chart

In [ ]:
fig = px.bar(
    x=selected_companies.values,
    y=selected_companies.index,
    orientation='h',
    title='Selected Portfolio (High Return & Low Risk)',
    labels={'x':'Annual ROI (%)','y':'Companies'}
)
fig.show()

## 10. Risk vs Return Scatter Plot

In [ ]:
fig = px.scatter(
    risk_return_df,
    x='Volatility (%)',
    y='Return (%)',
    text='Company',
    title='Risk vs Return Analysis'
)
fig.update_traces(textposition='top center')
fig.show()

## 11. Portfolio Allocation

In [ ]:
weights = np.repeat(1/len(selected_companies), len(selected_companies))
portfolio_df = pd.DataFrame({
    'Company': selected_companies.index,
    'Weight': weights
})
portfolio_df

## 12. Portfolio Allocation Pie Chart

In [ ]:
fig = px.pie(
    portfolio_df,
    values='Weight',
    names='Company',
    title='Portfolio Allocation'
)
fig.show()

## 13. Correlation Heatmap

In [ ]:
corr_matrix = returns[selected_companies.index].corr()
fig = px.imshow(corr_matrix, text_auto=True, title='Correlation Heatmap')
fig.show()

## 14. Portfolio Expected Return

In [ ]:
portfolio_return = np.sum(selected_companies.values * weights)
portfolio_return

## 15. Portfolio Risk

In [ ]:
selected_returns = returns[selected_companies.index]
cov_matrix = selected_returns.cov() * 252
portfolio_volatility = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights))) * 100
portfolio_volatility

## 16. Portfolio Growth Over Time

In [ ]:
portfolio_daily_return = selected_returns.dot(weights)
portfolio_growth = (1 + portfolio_daily_return).cumprod()

fig = px.line(
    portfolio_growth,
    title='Portfolio Growth Over Time',
    labels={'value':'Portfolio Value','index':'Date'}
)
fig.show()

## 17. SIP Investment Simulation

In [ ]:
initial_monthly_investment = 5000
investment_years = [1,3,5,10]
avg_roi = portfolio_return / 100
n = 12
annual_increase_rate = 0.10

## 18. Future Value Function

In [ ]:
def future_value_with_growth(initial_P, r, n, years, growth_rate):
    total_fv = 0
    for year in range(years):
        P = initial_P * ((1 + growth_rate) ** year)
        t = years - year
        fv = P * (((1 + r/n)**(n*t) - 1) / (r/n)) * (1 + r/n)
        total_fv += fv
    return total_fv

## 19. Calculate SIP Future Value

In [ ]:
future_values = [
    future_value_with_growth(initial_monthly_investment, avg_roi, n, t, annual_increase_rate)
    for t in investment_years
]
future_values

## 20. SIP Growth Visualization

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=[str(i)+' Years' for i in investment_years],
    y=future_values,
    mode='lines+markers',
    name='Investment Growth'
))

fig.update_layout(
    title='SIP Growth Projection',
    xaxis_title='Investment Period',
    yaxis_title='Future Value (INR)'
)

fig.show()